# 08B | Improving crop classification in Burkina Faso

**Study area:** Koumbia • **Author:** Ibrahim Kabore

Notebook 08 established a six-crop baseline. Its optical histogram gradient boosting achieved 0.2704 validation macro F1. Here we investigate whether the timing of crop development contains useful information lost through seasonal mean pooling. We compare representations on identical observations, not on different subsets.

The 2022 test results have already been seen in Notebook 08. This experiment uses only 2017, 2018 and 2020 for training and 2021 for development validation. A new independent assessment is needed after model development. No 2022 scores are calculated here.

## 1. Environment and shared pretrained encoder

Run on a Colab GPU. The cached checkpoint is the global pretrained encoder, not the French crop classifier. Its weights remain frozen.

In [ ]:
import sys, subprocess
REVISION = "bf8667902f583ec71fcca6b1a7bd1e48f64000f7"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    f"worldcereal @ git+https://github.com/WorldCereal/worldcereal-classification.git@{REVISION}"])

In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, zipfile, io, requests
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from importlib.metadata import version
from prometheo.models import Presto
from prometheo.models.pooling import PoolingMethods
from prometheo.predictors import Predictors, S1_BANDS, S2_BANDS, METEO_BANDS, DEM_BANDS, NODATAVALUE
from worldcereal.openeo.parameters import DEFAULT_SEASONAL_MODEL_URL
from worldcereal.utils.models import load_model_artifact

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ESA_EO_Projects")
PROJECT = ROOT / "WorldCereal_Burkina_Faso"
DATA = PROJECT / "data"
OUT = PROJECT / "outputs"
MODELS = PROJECT / "models"
for folder in [OUT, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Select a GPU runtime in Colab"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| PromethEO:", version("prometheo"),
      "| WorldCereal:", version("worldcereal"))
assert (DATA / "burkina_extractions_clean.parquet").is_file()
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from scipy.spatial import cKDTree
from pyproj import Transformer
import sklearn, joblib
RESULTS = OUT / "improvement"
MODEL_DIR = MODELS / "improvement"
for folder in [RESULTS, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42


In [ ]:
EXPECTED_SHA256 = "e16dce35b55b368c1553dd6c6dfe26f39d27818f24a9d710e2922cd4f00fed88"
MODEL_URL = DEFAULT_SEASONAL_MODEL_URL
assert "202608211601.zip" in MODEL_URL, MODEL_URL
french_bundle = ROOT / "WorldCereal_Crop_Mapping/models/2026/global_seasonal_20260821.zip"
bundle = french_bundle if french_bundle.is_file() else MODELS / "global_seasonal_20260821.zip"
if not bundle.is_file():
    r = requests.get(MODEL_URL, timeout=(30,120))
    r.raise_for_status()
    assert zipfile.is_zipfile(io.BytesIO(r.content))
    bundle.write_bytes(r.content)
artifact = load_model_artifact(bundle, cache_root=Path("/content/wc_cache"), encoder_only=True)
config = artifact.manifest
encoder_path = artifact.extract_dir / config["artifacts"]["checkpoints"]["encoder_only"]
fingerprint = hashlib.sha256(encoder_path.read_bytes()).hexdigest()
assert fingerprint == EXPECTED_SHA256, f"Unexpected encoder: {fingerprint}"
declared = (config.get("backbone") or {}).get("fingerprint")
if declared:
    assert fingerprint == declared
encoder = Presto()
encoder.load_state_dict(torch.load(encoder_path, map_location="cpu", weights_only=True), strict=True)
encoder = encoder.to("cuda").eval()
encoder.requires_grad_(False)
print("Verified:", fingerprint)
print("Checkpoint:", bundle)

## 2. Reconstruct the matched development dataset

The same reference IDs, input bands, monthly quality criteria and temporal partitions as Notebook 07 are retained. Sentinel-1 is masked throughout the new representations because radar was absent in 2020 and 2021.

In [ ]:
KEYS = ["ref_id","sample_id"]
META = ["label","label_year","split","spatial_group","lon","lat","ewoc_code"]
BAND_MAP = {**{f"S2-L2A-{b}": b.replace("B0","B") for b in
                ["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]},
    "S1-SIGMA0-VV":"VV", "S1-SIGMA0-VH":"VH",
    "AGERA5-PRECIP":"precipitation", "AGERA5-TMEAN":"temperature",
    "elevation":"elevation", "slope":"slope"}
BANDS = list(BAND_MAP)
FEATURES = [f"presto_ft_{i}" for i in range(128)]
POOL_MONTHS = [5,6,7,8,9,10]
BATCH = 256
raw = pd.read_parquet(DATA / "burkina_extractions_clean.parquet", filters=[("split","in",["train","validation"])])
assert set(KEYS + META + BANDS + ["timestamp"]) <= set(raw.columns)
assert raw[KEYS + META].notna().all().all()
raw["timestamp"] = pd.to_datetime(raw.timestamp, utc=True).dt.tz_localize(None)
raw["month"] = raw.timestamp.dt.month
raw["label_year"] = raw.label_year.astype(int)
assert raw.timestamp.dt.year.eq(raw.label_year).all()
assert not raw.duplicated(KEYS + ["month"]).any()
assert raw.groupby(KEYS).month.nunique().eq(12).all()
assert raw.groupby(KEYS)[META].nunique().le(1).all().all()
assert np.isfinite(raw[["lon","lat"]]).all().all()
expected_split = {2017:"train",2018:"train",2020:"train",2021:"validation",2022:"test"}
assert raw.split.eq(raw.label_year.map(expected_split)).all()
print("Rows:",len(raw))
print("Samples:", len(raw[KEYS].drop_duplicates()))
display(pd.crosstab(raw.drop_duplicates(KEYS).label,raw.drop_duplicates(KEYS).split,margins=True))

In [ ]:
raw[BANDS] = raw[BANDS].apply(pd.to_numeric, errors="coerce").replace([65535, np.inf, -np.inf],np.nan)
meta = raw.drop_duplicates(KEYS).set_index(KEYS)[META].sort_index()
wide = raw.pivot(index=KEYS,columns="month",values=BANDS).reindex(
    index=meta.index,columns=pd.MultiIndex.from_product([BANDS,range(1,13)]))
values = {b:wide[b].to_numpy(dtype=np.float32) for b in BANDS}
display(pd.DataFrame({b:pd.Series(a.ravel()).describe() for b,a in values.items()}).T)
optical = np.isfinite(values["S2-L2A-B08"])
radar = np.isfinite(values["S1-SIGMA0-VV"]) & (values["S1-SIGMA0-VV"]>0)
keep = (optical.sum(1)>=6) & (optical[:,4:10].sum(1)>=3)
print("Before common quality filter:",len(meta), "| retained:",int(keep.sum()))
meta = meta.iloc[np.flatnonzero(keep)].reset_index()
values = {b:a[keep] for b,a in values.items()}
meta["optical_months"] = optical[keep].sum(1)
meta["radar_months"] = radar[keep].sum(1)
display(pd.crosstab(meta.label,meta.split,margins=True))
assert set(meta.label)=={"Maize","Cotton","Sorghum","Soybean","Cowpea","Groundnut"}
assert (pd.crosstab(meta.label,meta.split)>0).all().all()
display(meta.groupby("label_year")[["optical_months","radar_months"]].mean().round(2))
assert set(meta.split)=={"train","validation"}
train_idx = np.flatnonzero(meta.split.eq("train"))
val_idx = np.flatnonzero(meta.split.eq("validation"))
Y = meta.label.to_numpy()
CLASSES = sorted(set(Y))
assert len(train_idx)==992 and len(val_idx)==271


## 3. Explore crop phenology

NDVI measures vegetation greenness from red and near-infrared reflectance. The lines show median monthly NDVI and shaded bands show the interquartile range. Overlapping profiles suggest why different crops can be confused; these are descriptive plots, not proof of separability.

In [ ]:
def ratio(a,b):
    return np.divide(a-b,a+b,out=np.full_like(a,np.nan),where=np.isfinite(a+b)&((a+b)!=0))
ndvi = ratio(values["S2-L2A-B08"],values["S2-L2A-B04"])
fig,axes = plt.subplots(2,3,figsize=(14,7),sharex=True,sharey=True)
for ax,label in zip(axes.ravel(),CLASSES):
    for name,idx,color in [("Train",train_idx,"tab:blue"),("2021 validation",val_idx,"tab:orange")]:
        d = ndvi[idx[Y[idx]==label]]
        q = np.nanpercentile(d,[25,50,75],axis=0)
        ax.plot(range(1,13),q[1],label=name,color=color)
        ax.fill_between(range(1,13),q[0],q[2],alpha=.15,color=color)
    ax.axvspan(5,10,color="green",alpha=.07)
    ax.set(title=label,xticks=range(1,13),xlabel="Month",ylabel="NDVI")
axes[0,0].legend()
fig.tight_layout()
fig.savefig(RESULTS/"crop_ndvi_profiles.png",dpi=180)
plt.show()

## 4. Retain monthly Presto representations

The encoder sees January to December in every run. Rather than averaging all six growing-season outputs immediately, we preserve the contextualised May to October vectors. These are not independent single-month inference calls.

In [ ]:
def encode(values, lon, lat, dates, use_radar):
    n,t = len(lon),len(dates)
    assert t==12
    s1 = np.full((n,1,1,t,len(S1_BANDS)),NODATAVALUE,np.float32)
    s2 = np.full((n,1,1,t,len(S2_BANDS)),NODATAVALUE,np.float32)
    meteo = np.full((n,1,1,t,len(METEO_BANDS)),NODATAVALUE,np.float32)
    dem = np.full((n,1,1,len(DEM_BANDS)),NODATAVALUE,np.float32)

    for source,band in BAND_MAP.items():
        a = np.asarray(values[source],dtype=np.float32).copy()
        assert a.shape==(n,t)
        ok = np.isfinite(a) & (a!=NODATAVALUE) & (a!=65535)
        a[~ok] = NODATAVALUE
        if band in S2_BANDS:
            if band=="B8A":
                a[:] = NODATAVALUE
            s2[:,0,0,:,S2_BANDS.index(band)] = a
        elif band in S1_BANDS:
            if use_radar:
                ok &= a>0
                a[~ok] = NODATAVALUE
                a[ok] = 20*np.log10(a[ok])-83
                s1[:,0,0,:,S1_BANDS.index(band)] = a
        elif band in METEO_BANDS:
            a[ok] /= 100000 if band=="precipitation" else 100
            meteo[:,0,0,:,METEO_BANDS.index(band)] = a
        elif band in DEM_BANDS:
            dem[:,0,0,DEM_BANDS.index(band)] = a[:,0]
        else:
            raise ValueError(band)

    dates = pd.DatetimeIndex(dates)
    stamps = np.stack([dates.day,dates.month,dates.year],axis=1)
    predictor = Predictors(s1=s1,s2=s2,meteo=meteo,dem=dem,
        latlon=np.stack([lat,lon],axis=1).astype("float32").reshape(n,1,1,2),
        timestamps=np.broadcast_to(stamps,(n,t,3)).copy())
    with torch.inference_mode():
        z = encoder(predictor,eval_pooling=PoolingMethods.TIME).reshape(n,t,128)
        keep = np.flatnonzero(dates.month.isin(POOL_MONTHS)).tolist()
        return z[:,keep,:].cpu().numpy()

In [ ]:
def run_batch(ix, use_radar):
    yrs = meta.label_year.to_numpy()[ix]
    assert len(set(yrs))==1
    dates = pd.date_range(f"{yrs[0]}-01-01",periods=12,freq="MS")
    return encode({b:a[ix] for b,a in values.items()},
                  meta.lon.to_numpy()[ix],meta.lat.to_numpy()[ix],dates,use_radar)

Z = np.empty((len(meta),6,128),dtype=np.float32)
for year in sorted(meta.label_year.unique()):
    ids = np.flatnonzero(meta.label_year.eq(year))
    for start in tqdm(range(0,len(ids),BATCH),desc=str(year)):
        ix = ids[start:start+BATCH]
        Z[ix] = run_batch(ix,False)
assert np.isfinite(Z).all()
print("Monthly vectors:",Z.shape)
seasonal = Z.mean(axis=1)
old = pd.read_parquet(OUT/"reference_embeddings_optical.parquet",filters=[("split","in",["train","validation"])])
old = old.set_index(KEYS).loc[pd.MultiIndex.from_frame(meta[KEYS])]
assert np.array_equal(old.label.to_numpy(),Y)
np.testing.assert_allclose(seasonal,old[FEATURES].to_numpy(),rtol=1e-4,atol=1e-5)
print("Seasonal vectors agree with Notebook 07.")


## 5. Construct competing feature sets

Seasonal mean has 128 features. Three two-month stages preserve early, middle and late season information (384 features). Monthly vectors preserve the complete May to October sequence (768 features). A conventional baseline uses the same optical, weather and terrain inputs without Presto. These fixed stages are a working hypothesis; they do not establish actual sowing or harvesting dates.

In [ ]:
X = {"seasonal":seasonal,"stages":Z.reshape(-1,3,2,128).mean(axis=2).reshape(-1,384),
     "monthly":Z.reshape(-1,768)}
names = {"seasonal":FEATURES,
         "stages":[f"{s}_ft_{i}" for s in ["may_jun","jul_aug","sep_oct"] for i in range(128)],
         "monthly":[f"m{m:02d}_ft_{i}" for m in POOL_MONTHS for i in range(128)]}
spectral = {b:values[b]/10000 for b in BANDS if b.startswith("S2") and not b.endswith("B8A")}
spectral.update({"NDVI":ndvi,"NDMI":ratio(values["S2-L2A-B08"],values["S2-L2A-B11"]),
                 "NDRE":ratio(values["S2-L2A-B08"],values["S2-L2A-B05"]),
                 "precipitation":values["AGERA5-PRECIP"]/100000,
                 "temperature":values["AGERA5-TMEAN"]/100})
X["spectral"] = np.column_stack([a[:,4:10] for a in spectral.values()]+[values[b][:,0] for b in ["elevation","slope"]])
names["spectral"] = [f"{b}_m{m:02d}" for b in spectral for m in POOL_MONTHS]+["elevation","slope"]
for name,arr in X.items():
    assert arr.shape==(len(meta),len(names[name]))
    print(name,arr.shape)


## 6. Visualise the embedding space

PCA is fitted solely on training seasonal features, then used to project both development partitions. Two components cannot represent all 128 dimensions, so visual overlap is not a definitive classification result.

In [ ]:
scaler = StandardScaler().fit(seasonal[train_idx])
pca = PCA(n_components=2).fit(scaler.transform(seasonal[train_idx]))
P = pca.transform(scaler.transform(seasonal))
fig,axes = plt.subplots(1,2,figsize=(12,5),sharex=True,sharey=True)
for ax,idx,title in zip(axes,[train_idx,val_idx],["Training years","2021 validation"]):
    for label in CLASSES:
        j = idx[Y[idx]==label]
        ax.scatter(P[j,0],P[j,1],s=12,alpha=.6,label=label)
    ax.set(title=title,xlabel="PC1",ylabel="PC2")
axes[1].legend()
fig.tight_layout()
fig.savefig(RESULTS/"seasonal_pca.png",dpi=180)
plt.show()
print("Explained variance:",pca.explained_variance_ratio_)

## 7. Compare classifiers on the development years

Use the original RF and HGB settings, plus regularised balanced logistic regression. Median imputation and standardisation for the conventional baseline and logistic regression are fitted within each training partition. Macro F1 gives all six crops equal weight. The 2021 validation comparison is exploratory because this year was also used for Notebook 08.

In [ ]:
def candidate(name,model):
    estimators = {
        "random_forest":RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight="balanced_subsample",n_jobs=2,random_state=SEED),
        "hist_gradient_boosting":HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=31,learning_rate=.1,random_state=SEED),
        "logistic_regression":LogisticRegression(C=1,class_weight="balanced",max_iter=2500,random_state=SEED)
    }
    steps = []
    if name=="spectral":
        steps.append(SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True))
    if model=="logistic_regression":
        steps.append(StandardScaler())
    return make_pipeline(*steps,estimators[model])
def metrics(y,p,classes=CLASSES):
    return {"accuracy":accuracy_score(y,p),"balanced_accuracy":balanced_accuracy_score(y,p),
            "macro_f1":f1_score(y,p,labels=classes,average="macro",zero_division=0)}
MODELNAMES = ["random_forest","hist_gradient_boosting","logistic_regression"]
rows, fitted = [], {}
for name,arr in X.items():
    for model in MODELNAMES:
        pipe = candidate(name,model).fit(arr[train_idx],Y[train_idx])
        fitted[name,model] = pipe
        rows.append({"configuration":name,"model":model,**metrics(Y[val_idx],pipe.predict(arr[val_idx]))})
validation = pd.DataFrame(rows).sort_values(["macro_f1","balanced_accuracy"],ascending=False)
display(validation.round(4))
validation.to_csv(RESULTS/"validation_metrics.csv",index=False)
ax = validation.pivot(index="configuration",columns="model",values="macro_f1").plot.bar(figsize=(11,4),rot=0)
ax.axhline(.2704,ls="--",color="black",label="Notebook 08 optical HGB")
ax.set(ylabel="Macro F1",title="2021 validation comparison",ylim=(0,1))
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS/"validation_comparison.png",dpi=180)
plt.show()


## 8. Spatial and temporal robustness

Run five grouped folds within training years using 2 km spatial blocks and exclude training observations within 100 m of held-out points. Also hold out each entire training year in turn. The year-based folds test temporal transfer but are not spatially independent. All model fits exclude 2021 and 2022 during these diagnostics.

In [ ]:
groups = meta.spatial_group.to_numpy()
xx,yy = Transformer.from_crs(4326,32630,always_xy=True).transform(meta.lon.to_numpy(),meta.lat.to_numpy())
xy = np.column_stack([xx,yy])
folds = []
cv = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
for fold,(tr,te) in enumerate(cv.split(X["seasonal"][train_idx],Y[train_idx],groups[train_idx]),1):
    ti,hi = train_idx[tr],train_idx[te]
    assert set(groups[ti]).isdisjoint(groups[hi])
    dist,_ = cKDTree(xy[hi]).query(xy[ti])
    folds.append(("spatial_100m",str(fold),ti[dist>100],hi))
for year in sorted(meta.label_year.iloc[train_idx].unique()):
    hi = train_idx[meta.label_year.iloc[train_idx].to_numpy()==year]
    ti = np.setdiff1d(train_idx,hi)
    folds.append(("leave_year_out",str(year),ti,hi))
cv_rows, audit = [], []
for scheme,fold,ti,hi in tqdm(folds):
    complete = set(Y[ti])==set(CLASSES) and set(Y[hi])==set(CLASSES)
    audit.append({"scheme":scheme,"fold":fold,"train":len(ti),"held_out":len(hi),"complete":complete})
    if not complete:
        print("Skipped incomplete fold:",scheme,fold)
        continue
    for name,arr in X.items():
        for model in MODELNAMES:
            pipe = candidate(name,model).fit(arr[ti],Y[ti])
            cv_rows.append({"scheme":scheme,"fold":fold,"configuration":name,"model":model,
                            **metrics(Y[hi],pipe.predict(arr[hi]))})
pd.DataFrame(audit).to_csv(RESULTS/"fold_audit.csv",index=False)
cv_scores = pd.DataFrame(cv_rows)
cv_scores.to_csv(RESULTS/"cv_metrics.csv",index=False)
summary = cv_scores.groupby(["scheme","configuration","model"])[["accuracy","balanced_accuracy","macro_f1"]].agg(["mean","std","count"])
display(summary.round(4))
summary.to_csv(RESULTS/"cv_summary.csv")


## 9. Inspect the provisional six-crop winner

Rank by 2021 macro F1. Review the spatial and temporal diagnostics before considering deployment. The winner remains a development candidate, not a validated operational model.

In [ ]:
winner = validation.iloc[0]
name,model = winner.configuration,winner.model
selected = fitted[name,model]
pred = selected.predict(X[name][val_idx])
report = pd.DataFrame(classification_report(Y[val_idx],pred,labels=CLASSES,output_dict=True,zero_division=0)).T
display(report.round(3))
report.to_csv(RESULTS/"selected_validation_report.csv")
fig,ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(Y[val_idx],pred,labels=CLASSES,normalize="true",cmap="Blues",values_format=".2f",xticks_rotation=45,ax=ax)
ax.grid(False)
ax.set_title("2021 validation | selected candidate")
fig.tight_layout()
fig.savefig(RESULTS/"selected_validation_confusion.png",dpi=180)
plt.show()
print("Selected:",name,model,"| macro F1:",round(winner.macro_f1,4))


## 10. Optional separate three-group classification

Cereals group maize and sorghum. Legumes group cowpea, groundnut and soybean. Cotton remains a fibre crop. This changes the scientific question and must not be compared directly against six-class F1. The groups refer to botanical legumes, not the EWOC commodity hierarchy.

In [ ]:
GROUP_MAP = {"Maize":"Cereals","Sorghum":"Cereals","Cowpea":"Legumes","Groundnut":"Legumes","Soybean":"Legumes","Cotton":"Cotton"}
YG = np.array([GROUP_MAP[y] for y in Y])
grouped = candidate(name,model).fit(X[name][train_idx],YG[train_idx])
gp = grouped.predict(X[name][val_idx])
group_classes = sorted(set(YG))
group_report = pd.DataFrame(classification_report(YG[val_idx],gp,labels=group_classes,output_dict=True,zero_division=0)).T
display(group_report.round(3))
group_report.to_csv(RESULTS/"three_group_validation_report.csv")
print(metrics(YG[val_idx],gp,group_classes))


## 11. Save development outputs

Store the selected pipeline and feature contract separately from Notebook 08. Notebook 09 must reproduce this exact feature representation before mapping. A regional map requires independent validation and a carefully verified cropland mask; a six-class classifier cannot recognise crops or land cover omitted from training.

In [ ]:
path = MODEL_DIR/"selected_classifier.joblib"
joblib.dump(selected,path)
manifest = {"configuration":name,"model":model,"feature_names":names[name],
            "checkpoint_sha256":fingerprint,"worldcereal_revision":REVISION,
            "worldcereal":version("worldcereal"),"prometheo":version("prometheo"),
            "sklearn_version":sklearn.__version__,"context":"Jan-Dec","pool_months":POOL_MONTHS,
            "use_radar":False,"mask_b8a":True,"fit_years":[2017,2018,2020],
            "validation_year":2021,"test_year":2022,"test_used_in_this_notebook":False,
            "validation_macro_f1":float(winner.macro_f1),"selection_metric":"validation_macro_f1",
            "classifier_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
            "classes":selected.classes_.tolist(),"crop_codes":{str(i+1):v for i,v in enumerate(selected.classes_)},
            "quality_filter":"B08 >=6 annual months and >=3 May-Oct months",
            "feature_contract":"See section 5 of Notebook 08B"}
(MODEL_DIR/"classifier_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Saved:",path)
print("Diagnostics:",RESULTS)
print("Notebook 08 baseline unchanged.")
